In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/spotify-tracks-dataset-detailed.csv")

df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
df.shape

(114000, 20)

In [4]:
df.columns.tolist()

['track_id',
 'artists',
 'album_name',
 'track_name',
 'popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'track_genre']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  object 
 1   artists           113999 non-null  object 
 2   album_name        113999 non-null  object 
 3   track_name        113999 non-null  object 
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           11

In [6]:
df.isnull().sum()

track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(450)

In [8]:
df[["track_name", "artists"]].duplicated().sum()

np.int64(32656)

In [9]:
df["track_name"].isnull().sum()

np.int64(1)

In [10]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=["track_name", "artists"])

df_clean = df_clean.drop_duplicates()

df_clean = df_clean.drop_duplicates(
    subset=["track_name", "artists"],
    keep="first"
)

df_clean = df_clean.reset_index(drop=True)

df_clean.shape

(81343, 20)

In [11]:
df_clean.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [12]:
df_clean[["track_name", "artists"]].duplicated().sum()

np.int64(0)

In [13]:
df_clean.to_csv(
    "../data/spotify_tracks_clean.csv",
    index=False
)

In [14]:
df_clean.shape

(81343, 20)

In [15]:
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

In [16]:
X = df_clean[features]

X.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.3580,0.715,87.917
1,0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.1010,0.267,77.489
2,0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.1170,0.120,76.332
3,0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.1320,0.143,181.740
4,0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.0829,0.167,119.949


In [17]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled.shape

(81343, 9)

In [27]:
def recommend_songs(
    song_label,
    n_recommendations=5
):
    matches = df_clean[
        df_clean["song_label"] == song_label
    ]

    if matches.empty:
        return f"Song '{song_label}' not found."

    song_index = matches.index[0]

    song_genre = df_clean.loc[
        song_index,
        "track_genre"
    ]

    song_vector = (
        X_scaled[song_index]
        .reshape(1, -1)
    )

    similarities = cosine_similarity(
        song_vector,
        X_scaled
    ).flatten()

    # Prefer songs from the same genre
    genre_mask = (
        df_clean["track_genre"]
        == song_genre
    )

    candidate_df = (
        df_clean[genre_mask]
        .copy()
    )

    candidate_indices = (
        candidate_df.index
    )

    candidate_df["similarity"] = (
        similarities[
            candidate_indices
        ]
    )

    # Normalize popularity to 0–1
    candidate_df["popularity_score"] = (
        candidate_df["popularity"]
        / 100
    )

    # 85% audio similarity
    # 15% popularity
    candidate_df["final_score"] = (
        0.85
        * candidate_df["similarity"]
        +
        0.15
        * candidate_df["popularity_score"]
    )

    # Don't recommend the selected song itself
    candidate_df = candidate_df[
        candidate_df["song_label"]
        != song_label
    ]

    recommendations = (
        candidate_df
        .sort_values(
            "final_score",
            ascending=False
        )
        .head(n_recommendations)
    )

    return recommendations[
        [
            "track_name",
            "artists",
            "track_genre",
            "popularity",
            "similarity",
            "final_score"
        ]
    ].reset_index(drop=True)

In [28]:
df_clean[
    df_clean["track_name"].str.contains(
        "Blinding Lights",
        case=False,
        na=False
    )
][
    [
        "song_label",
        "track_genre",
        "popularity"
    ]
]

,song_label,track_genre,popularity
12124,Blinding Lights — Kidz Bop Kids,children,0
19745,Blinding Lights — Lucas Estrada;Twan Ray,deep-house,51
25186,Blinding Lights — Revelries;Victoria Voss,edm,64
36791,Blinding Lights (Instrumental Guitar) — Guus D...,guitar,43
61238,Blinding Lights — The Weeknd,pop,91
63603,Blinding Lights — All Time Low,punk-rock,1


In [29]:
recommend_songs(
    "Blinding Lights — The Weeknd",
    5
)

,track_name,artists,track_genre,popularity,similarity,final_score
0,MIDDLE OF THE NIGHT,Elley Duhé,pop,90,0.895576,0.896239
1,Wildest Dreams,Taylor Swift,pop,80,0.891646,0.877899
2,"Humraah (From ""Malang - Unleash The Madness"")",Sachet Tandon;The Fusion Project,pop,67,0.892269,0.858929
3,Shiddat Title Track,Manan Bhardwaj,pop,65,0.874038,0.840432
4,Orasaadha - Madras Gig,Vivek - Mervin,pop,67,0.820997,0.798348


In [30]:
df_clean[
    df_clean["track_name"].str.contains(
        "Wildest Dreams",
        case=False,
        na=False
    )
][
    [
        "song_label",
        "track_genre",
        "popularity"
    ]
]

,song_label,track_genre,popularity
7125,Never in My Wildest Dreams — Dan Auerbach,blues,64
61426,Wildest Dreams — Taylor Swift,pop,80
63417,In Your Wildest Dreams — The Reverend Horton Heat,psych-rock,27


In [31]:
recommend_songs(
    "Wildest Dreams — Taylor Swift",
    5
)

,track_name,artists,track_genre,popularity,similarity,final_score
0,Blinding Lights,The Weeknd,pop,91,0.891646,0.894399
1,Die For You,The Weeknd,pop,88,0.857731,0.861071
2,Orasaadha - Madras Gig,Vivek - Mervin,pop,67,0.830480,0.806408
3,Shiddat Title Track,Manan Bhardwaj,pop,65,0.780656,0.761058
4,Drag Me Down,One Direction,pop,80,0.741650,0.750402
